# SPLADE++ (2021)
---
[[paper]](https://arxiv.org/pdf/2004.04906)<br>
SPLADE++ = **S**parse **P**rojection of **LA**nguage Mo**DE**ls++

SPLADE++ — это нейронный метод информационного поиска, который генерирует разреженные (sparse) лексикализованные представления как для запросов, так и для документов, используя один BERT-подобный энкодер. Он позволяет совместить преимущества семантического понимания нейронных моделей с эффективностью и интерпретируемостью традиционных методов разреженного поиска на основе инвертированных индексов.

### Контекст
В области информационного поиска традиционно доминировали **разреженные методы** (sparse retrieval), такие как BM25 (поздние 90-е) и TF-IDF. Они обладают высокой скоростью, отличной масштабируемостью за счет использования инвертированных индексов и хорошей интерпретируемостью, поскольку совпадения основаны на конкретных словах. Однако их главный недостаток — **лексическое несоответствие** (lexical mismatch): они не могут находить документы, использующие синонимы или схожие по смыслу, но разные по лексике слова.

Появление трансформерных моделей привело к расцвету **плотных методов** (dense retrieval), таких как DPR (2020) и ANCE (2020). Эти методы обучают нейронные сети для отображения запросов и документов в низкоразмерные векторы (dense embeddings), где семантически похожие объекты оказываются ближе в векторном пространстве. Плотные методы демонстрируют превосходное семантическое понимание и хорошо справляются с лексическим несоответствием. Однако они имеют свои недостатки:
*   **Производительность:** Требуют специализированных алгоритмов поиска ближайших соседей (ANN) для высокоразмерных векторов, что может быть медленнее, чем поиск по инвертированному индексу.
*   **Масштабируемость:** Индексы плотных векторов могут быть большими, а их обновление сложным.
*   **Интерпретируемость:** Плотные векторы являются "черным ящиком", и понять, почему тот или иной документ был релевантен, гораздо сложнее.

### Идея
SPLADE++ был разработан для объединения сильных сторон разреженного и плотного поиска. Вместо того чтобы генерировать плотные векторы, модель создает **разреженные, взвешенные лексикализованные векторы**, где каждый элемент вектора соответствует слову из словаря (или его токену), а значение элемента показывает его "важность" для документа/запроса. Это позволяет:
1.  Использовать **инвертированные индексы** и стандартные поисковые движки (например, Lucene, Elasticsearch), что обеспечивает высокую скорость и масштабируемость.
2.  Сохранить **семантическое понимание** нейронных моделей, поскольку веса "слов" назначаются контекстно-зависимо.
3.  Обеспечить **интерпретируемость**, так как результирующие векторы явно показывают, какие слова (токены) являются наиболее важными для представления документа или запроса.

Ключевая новизна SPLADE++ по сравнению с предшествующими подходами и даже оригинальным SPLADE (2021) заключается в:
*   Использовании **контрастивного обучения** для генерации этих разреженных представлений, что значительно улучшает их качество по сравнению с более простыми self-supervised задачами.
*   Применении техники **DocT5Query-style augmentation**, где для каждого документа генерируются синтетические запросы, что обогащает лексическое представление документов и предоставляет больше тренировочных пар.

### Постановка задачи
Дана коллекция текстовых документов $D = \{d_1, d_2, \ldots, d_N\}$ и запрос $Q$. Задача состоит в поиске топ-K наиболее релевантных документов из коллекции $D$ для заданного запроса $Q$.

### Альтернативные методы на момент появления SPLADE++

На момент появления SPLADE++ существовали следующие категории методов:

1.  **Традиционный разреженный поиск:**
    *   **BM25** (поздние 90-е): Использует частоту терминов, длину документа и инвертированную частоту документа для ранжирования. Быстрый, масштабируемый, но страдает от лексического несоответствия.
    *   **TF-IDF:** Более простая предшественница BM25, также основанная на частотах слов.

2.  **Плотный поиск (Dense Retrieval):**
    *   **Word2Vec** (2013), **DSSM** (2013): Ранние попытки отобразить текст в плотные векторы, но с ограниченным семантическим пониманием или фокусировкой на словах.
    *   **DrQA** (2017): Использовал двухбашенные LSTM-модели, обученные self-supervised методом (Inverse Cloze Task).
    *   **DPR** (2020): Модель на основе двух BERT-энкодеров, обученная с использованием контрастивного обучения, стала одним из золотых стандартов в плотном поиске.
    *   **ANCE** (2020): Улучшил DPR за счет асинхронного майнинга "сложных" негативных примеров.

3.  **Гибридные/предшествующие нейронные разреженные подходы:**
    *   **DocT5Query** (2020): Не является методом поиска, а скорее техникой **аугментации документов**. Он использует модель T5 для генерации потенциальных запросов для каждого документа. Затем эти сгенерированные запросы добавляются к исходному тексту документа, чтобы обогатить его лексически для последующего поиска традиционными разреженными методами. Этот подход сильно повлиял на дизайн SPLADE++.
    *   **Original SPLADE** (2021): Предшественник SPLADE++. Впервые предложил идею генерации разреженных лексикализованных представлений с использованием BERT-модели с функцией **SparseMax** и L1-регуляризацией, обученной на задаче, аналогичной Masked Language Modeling (MLM), но с маскированием токенов, которые должны быть активированы. SPLADE++ улучшает его, используя контрастивное обучение.

### Архитектура модели

Архитектура SPLADE++ относительно проста и основана на одном энкодере:

1.  **BERT-подобный энкодер:** Основой является предварительно обученная модель-трансформер (например, MiniLM, RoBERTa), которая принимает на вход последовательность токенов (запроса или документа).
2.  **Выходной слой:** Из скрытых состояний BERT-модели для каждого токена (например, [CLS]-токен или усреднение всех токенов) затем генерируется вектор размерности словаря модели. Этот вектор представляет собой **лексикализованное представление**.
3.  **SparseMax активация:** Для обеспечения разреженности, полученный вектор проходит через функцию **SparseMax** (вместо обычной Softmax). SparseMax гарантирует, что только часть элементов вектора будут иметь ненулевые значения, остальные будут обнулены. Это критически важно для создания разреженных векторов.
4.  **Max-Pooling:** После SparseMax, для каждого токена в последовательности мы получаем разреженный вектор. Затем применяется max-pooling (или другой вид pooling'а) по всем токенам последовательности. Это означает, что для каждого слова из словаря мы берем максимальное значение активации, которое оно получило на какой-либо позиции в исходном тексте.
5.  **Разреженный вектор:** Конечный выход — это разреженный, высокоразмерный вектор, где каждый элемент соответствует слову (токену) из словаря модели. Ненулевые значения элементов представляют собой **веса** или "важность" соответствующих слов для данного запроса или документа.

### Алгоритм обучения

Обучение SPLADE++ базируется на **контрастивном обучении** и значительно отличается от оригинального SPLADE, который использовал MLM-подобную задачу.

1.  **Подготовка данных:** Для каждой обучающей итерации выбирается тройка: `(запрос Q, позитивный документ D+, негативный документ D-)`.
    *   **Позитивные пары (Q, D+):** Берутся из существующих размеченных датасетов (например, MS MARCO).
    *   **Негативные примеры (D-):** Генерируются разными способами:
        *   **In-batch negatives:** Другие документы в том же батче считаются негативными.
        *   **Hard negatives:** Специально майнятся "сложные" негативные примеры, которые семантически близки к запросу, но нерелевантны, чтобы модель училась лучше их различать (например, с помощью BM25 или других нейронных моделей).
    *   **DocT5Query-стиль аугментация:** Для каждого документа в коллекции, модель T5 генерирует несколько потенциальных запросов. Эти сгенерированные запросы затем используются как "фиктивные" запросы для обучения, расширяя данные и делая представления документов более насыщенными лексически.

2.  **Процесс обучения:**
    *   Все три элемента тройки (`Q`, `D+`, `D-`) по очереди проходят через **общий энкодер SPLADE++**.
    *   На выходе для каждого элемента получается свой **разреженный вектор** (например, $v_Q$, $v_{D+}$, $v_{D-}$).
    *   **Вычисление релевантности:** Оценка релевантности между запросом и документом вычисляется как **скалярное произведение** их разреженных векторов: $score(Q, D) = v_Q \cdot v_D$.
    *   **Функция потерь:** Используется **Negative Log-Likelihood (NLL)** на основе softmax-нормализованных оценок релевантности. Цель — максимизировать оценку для позитивной пары $(Q, D+)$ и минимизировать для негативных $(Q, D-)$.
        $Loss = -\log \frac{\exp(score(Q, D^+))}{\exp(score(Q, D^+)) + \sum_{D^- \in Neg} \exp(score(Q, D^-))}$
    *   **L1-регуляризация:** К функции потерь добавляется L1-регуляризация для разреженных векторов. Это принуждает модель минимизировать количество ненулевых элементов, тем самым усиливая разреженность и фокусируясь только на наиболее значимых словах.
        $TotalLoss = Loss + \lambda \sum_{i} |v_{D,i}| + \lambda \sum_{j} |v_{Q,j}|$
        (где $\lambda$ — коэффициент регуляризации)

3.  **Обновление весов:** Градиенты вычисляются и используются для обновления весов BERT-энкодера.

### Алгоритм инференса и индексации

1.  **Индексация документов (оффлайн):**
    *   Для каждого документа $d_i$ в коллекции:
        *   Пропустить $d_i$ через обученный энкодер SPLADE++.
        *   Получить разреженный вектор $v_{d_i}$ (список пар `(токен_ID, вес)`).
        *   Сохранить эти пары в **инвертированном индексе** (например, Apache Lucene, Elasticsearch). Каждый токен из словаря становится "термином", а вес — его "TF-IDF-подобным" значением.

2.  **Поиск по запросу (онлайн):**
    *   Когда приходит запрос $Q$:
        *   Пропустить $Q$ через обученный энкодер SPLADE++.
        *   Получить разреженный вектор $v_Q$ для запроса.
        *   Использовать $v_Q$ для выполнения поиска в инвертированном индексе.
        *   Для каждого найденного документа, вычислить **оценку релевантности** как скалярное произведение $v_Q \cdot v_D$.
        *   Ранжировать документы по этой оценке и вернуть топ-K.

### Результаты

SPLADE++ был тщательно протестирован на стандартных бенчмарках для информационного поиска, таких как **MS MARCO Passage Ranking** и **MS MARCO Document Ranking**.

*   **Эффективность:** SPLADE++ демонстрирует конкурентную или даже превосходящую эффективность по сравнению с передовыми моделями плотного поиска (DPR, ANCE) на основных метриках, таких как MRR@10 и Recall@K. Например, на MS MARCO Passage Ranking, SPLADE++ значительно повысил MRR@10 по сравнению с BM25 (более чем на 30 процентных пунктов) и приблизился к производительности DPR, иногда даже немного опережая его.
*   **Скорость и масштабируемость:** За счет использования инвертированных индексов, SPLADE++ позволяет использовать высокооптимизированные движки разреженного поиска, обеспечивая значительно более высокую скорость поиска по сравнению с методами плотного поиска, требующими ANN-индексов. Индексация и хранение разреженных векторов также более эффективны с точки зрения занимаемого места на диске и времени построения индекса.
*   **Интерпретируемость:** Ключевое преимущество SPLADE++. Сгенерированные разреженные векторы явно показывают, какие слова (токены) в запросе и документе были активированы и с какой силой. Это позволяет пользователям и разработчикам понимать логику ранжирования, что практически невозможно с плотными векторами. Например, можно вывести топ-K активированных токенов для запроса или документа, чтобы увидеть их семантическое "ядро".
*   **Гибкость:** SPLADE++ легко интегрируется в существующие инфраструктуры поиска на основе Lucene/Elasticsearch, что упрощает его внедрение в реальные системы.

В целом, SPLADE++ успешно демонстрирует, что можно получить семантическую мощь нейронных моделей, сохраняя при этом эффективность, масштабируемость и интерпретируемость традиционного разреженного поиска, создавая таким образом новое направление в нейронном поиске.

## 📝 Критический анализ

```markdown
# SPLADE++ (2021)
---
[[paper]](https://arxiv.org/pdf/2004.04906)<br>
SPLADE++ = **S**parse **P**rojection of **LA**nguage Mo**DE**ls++

SPLADE++ — нейронный метод информационного поиска, генерирующий разреженные лексикализованные представления для запросов и документов с использованием одного BERT-подобного энкодера. Он сочетает семантическое понимание нейронных моделей с эффективностью и интерпретируемостью разреженного поиска.

### Контекст
Традиционные разреженные методы, такие как BM25, обеспечивают скорость и интерпретируемость, но страдают от лексического несоответствия. Плотные методы, как DPR и ANCE, решают эту проблему, но требуют сложных алгоритмов поиска и менее интерпретируемы.

### Идея
SPLADE++ объединяет преимущества разреженного и плотного поиска, создавая разреженные, взвешенные лексикализованные векторы. Это позволяет использовать инвертированные индексы, сохраняя семантическое понимание и интерпретируемость.

<img src="img/img.png" width=500>

### Постановка задачи
Поиск топ-K релевантных документов для запроса $Q$ из коллекции $D$.

### Альтернативные методы
1. **Разреженный поиск:** BM25, TF-IDF.
2. **Плотный поиск:** Word2Vec, DSSM, DrQA, DPR, ANCE.
3. **Гибридные подходы:** DocT5Query, оригинальный SPLADE.

### Архитектура
- **BERT-подобный энкодер:** Генерирует лексикализованные представления.
- **SparseMax активация:** Обеспечивает разреженность векторов.
- **Max-Pooling:** Выбирает максимальные активации токенов.
- **Разреженный вектор:** Представляет важность слов.

### Алгоритм обучения
Используется контрастивное обучение с тройками `(Q, D+, D-)`. Позитивные пары берутся из размеченных датасетов, негативные генерируются. Применяется DocT5Query-style augmentation для обогащения данных. Потери вычисляются с помощью Negative Log-Likelihood и L1-регуляризации для усиления разреженности.

### Алгоритм инференса
1. **Индексация документов:** Генерация разреженных векторов и сохранение в инвертированном индексе.
2. **Поиск по запросу:** Генерация вектора для запроса, поиск в индексе и ранжирование документов.

### Результаты
- **Эффективность:** SPLADE++ превосходит BM25 и сопоставим с DPR по MRR@10 и Recall@K.
- **Скорость и масштабируемость:** Использование инвертированных индексов обеспечивает высокую скорость и эффективность хранения.
- **Интерпретируемость:** Разреженные векторы показывают важные слова, улучшая понимание логики ранжирования.
- **Гибкость:** Легко интегрируется в существующие системы на основе Lucene/Elasticsearch.

SPLADE++ демонстрирует, что можно объединить семантическую мощь нейронных моделей с эффективностью и интерпретируемостью разреженного поиска.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel

# SPLADE++ Model Implementation
class SPLADEPlusPlus(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased'):
        super(SPLADEPlusPlus, self).__init__()
        # Load a pre-trained BERT model
        self.bert = BertModel.from_pretrained(bert_model_name)
        # Define a linear layer to project BERT outputs to vocabulary size
        self.vocab_projection = nn.Linear(self.bert.config.hidden_size, self.bert.config.vocab_size)
    
    def forward(self, input_ids, attention_mask):
        # Get BERT outputs
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Use the hidden states of the last layer
        last_hidden_state = outputs.last_hidden_state
        # Project to vocabulary size
        vocab_logits = self.vocab_projection(last_hidden_state)
        # Apply SparseMax to ensure sparsity
        sparse_vocab_logits = self.sparsemax(vocab_logits)
        # Max pooling over the sequence length dimension
        sparse_representation = torch.max(sparse_vocab_logits, dim=1).values
        return sparse_representation

    def sparsemax(self, logits):
        # SparseMax activation function
        # This is a simplified version for illustration purposes
        sorted_logits, _ = torch.sort(logits, descending=True)
        cumsum_logits = torch.cumsum(sorted_logits, dim=-1)
        range_values = torch.arange(1, logits.size(-1) + 1, device=logits.device)
        threshold = (cumsum_logits - 1) / range_values
        is_gt = sorted_logits > threshold
        k = is_gt.sum(dim=-1, keepdim=True)
        tau = (cumsum_logits.gather(dim=-1, index=k - 1) - 1) / k
        return torch.max(logits - tau, torch.zeros_like(logits))

# Example usage
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = SPLADEPlusPlus()

# Sample query and document
query = "What is the capital of France?"
document = "Paris is the capital city of France."

# Tokenize inputs
query_inputs = tokenizer(query, return_tensors='pt', padding=True, truncation=True)
document_inputs = tokenizer(document, return_tensors='pt', padding=True, truncation=True)

# Get sparse representations
query_sparse_rep = model(query_inputs['input_ids'], query_inputs['attention_mask'])
document_sparse_rep = model(document_inputs['input_ids'], document_inputs['attention_mask'])

# Compute relevance score (dot product)
relevance_score = torch.dot(query_sparse_rep.squeeze(), document_sparse_rep.squeeze())

print(f"Relevance Score: {relevance_score.item()}")

# Note: This is a simplified illustration of SPLADE++ concepts.
# In practice, you would need to handle batching, negative sampling, and training loops.
```

### Key Concepts Illustrated:

1. **Sparse Representation**: The model generates sparse, lexicon-based vectors for queries and documents using a BERT-like encoder and a SparseMax activation function.

2. **SparseMax Activation**: This function ensures that only a subset of the vocabulary is activated, creating a sparse representation.

3. **Max-Pooling**: After applying SparseMax, max-pooling is used to aggregate token-level activations into a single vector representing the entire input.

4. **Relevance Scoring**: The relevance between a query and a document is computed using the dot product of their sparse representations.

This code provides a basic framework to understand the SPLADE++ method, focusing on its unique approach to generating sparse representations using a BERT-like model.